In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install imbalanced-learn -q

import h5py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler
import warnings
warnings.filterwarnings('ignore')

print("✅ Ready!")

Mounted at /content/drive
✅ Ready!


In [ ]:
DRIVE_PATH = '/content/drive/MyDrive/UML_Project/'

with h5py.File(DRIVE_PATH + 'semantic_embeddings.h5', 'r') as f:
    classification_emb = f['classification_embedding'][:]
    urls = f['URL'][:]

urls = [u.decode('utf-8') if isinstance(u, bytes) else u for u in urls]
print(f"✅ Embeddings shape: {classification_emb.shape}")
print(f"✅ URLs count: {len(urls)}")

✅ Embeddings shape: (8606, 768)
✅ URLs count: 8606


In [ ]:
df = pd.read_csv(DRIVE_PATH + 'EDA_data-FULL.csv')

valid_labels = ['update-me', 'give-me-perspective', 'educate-me', 'connect-me', 'inspire-me', 'help-me']
df_filtered = df[df['User_Needs'].isin(valid_labels)].copy().reset_index(drop=True)

emb_df = pd.DataFrame({'URL': urls, 'emb_index': range(len(urls))})
merged = emb_df.merge(df_filtered[['URL', 'User_Needs']], on='URL', how='inner')

print(f"✅ Matched: {len(merged)} articles")
print(merged['User_Needs'].value_counts())

✅ Matched: 3855 articles
User_Needs
update-me              1919
give-me-perspective     727
educate-me              586
connect-me              304
inspire-me              216
help-me                 103
Name: count, dtype: int64


In [ ]:
X = classification_emb[merged['emb_index'].values]
y_raw = merged['User_Needs'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

✅ Train: (9210, 768), Test: (771, 768)


In [ ]:
classifiers = {
    "SVM (RBF)": SVC(kernel='rbf', C=10, gamma='scale', random_state=42),
    "MLP Neural Net": MLPClassifier(
        hidden_layer_sizes=(512, 256, 128),
        activation='relu',
        max_iter=300,
        learning_rate_init=0.001,
        early_stopping=True,
        random_state=42
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=10.0,
        solver='lbfgs',
        multi_class='multinomial',
        random_state=42
    )
}

best_acc = 0
best_name = ""
best_pred = None

for name, clf in classifiers.items():
    print(f"Training {name}...")
    clf.fit(X_train_scaled, y_train_bal)
    pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    print(f"  → Accuracy: {acc*100:.1f}%")
    if acc > best_acc:
        best_acc = acc
        best_name = name
        best_pred = pred

print(f"\n{'='*50}")
print(f"  BEST: {best_name}")
print(f"  ACCURACY: {best_acc*100:.1f}%")
print(f"{'='*50}\n")
print(classification_report(y_test, best_pred, target_names=le.classes_))

Training SVM (RBF)...
  → Accuracy: 71.2%
Training MLP Neural Net...
  → Accuracy: 70.4%
Training Logistic Regression...
  → Accuracy: 62.9%

  BEST: SVM (RBF)
  ACCURACY: 71.2%

                     precision    recall  f1-score   support

         connect-me       0.47      0.57      0.51        61
         educate-me       0.49      0.37      0.42       117
give-me-perspective       0.76      0.68      0.72       145
            help-me       0.56      0.24      0.33        21
         inspire-me       0.57      0.56      0.56        43
          update-me       0.81      0.89      0.85       384

           accuracy                           0.71       771
          macro avg       0.61      0.55      0.57       771
       weighted avg       0.70      0.71      0.70       771

